
# Qwen2‑VL Image Captioning — FP16 vs 8‑bit (LLM vs LLM & Vision)

**Track**: Captioning (CIDEr and BLEU‑4)\
**Dataset**: COCO 2017 val (5k)\
**Model**: Qwen2‑VL 2B


### (1) Setup: Installs & Imports
In this section, we first prepare the environment by installing the necessary dependencies for the project. The following libraries are installed:

- **Transformers**, **Accelerate**, **Datasets**, and **Pillow**: These libraries are used for working with pre-trained models, accelerating computation, handling datasets, and image processing respectively.
- **BitsAndBytes**: This library is required for working with 8-bit quantization.
- **TorchMetrics** and **Evaluate**: These are used for performance evaluation metrics, such as BLEU scores.
- **Sacrebleu**: A library for evaluating translation quality.
- **pycocoevalcap**: Used for COCO evaluation metrics like CIDEr.
- **CodeCarbon**: This tool is used to estimate and track the carbon emissions during computation, promoting sustainability.


We proceed to import a variety of libraries and modules that will be used throughout the notebook. These include:

- **PIL (Pillow)**: A library for image processing.
- **Datasets**: To load datasets for training/evaluation.
- **tqdm**: For displaying progress bars during data processing.



In [ ]:

# --- Dependency prep ---
!pip -q install --upgrade pip
!pip -q install "transformers>=4.44" "accelerate>=0.33" "datasets>=2.19" pillow
!pip -q install bitsandbytes
!pip -q install torchmetrics evaluate sacrebleu
!pip -q install --use-pep517 git+https://github.com/salaniz/pycocoevalcap.git
!pip -q install codecarbon

!pip -q install --upgrade --force-reinstall "httpx[http2]==0.28.1" "httpcore>=1.0.2,<1.1.0"

import pkg_resources, sys
def _ver(pkg):
    try:
        return pkg_resources.get_distribution(pkg).version
    except Exception:
        return "not-installed"
print("httpx:", _ver("httpx"))
print("httpcore:", _ver("httpcore"))

import os, json, time, math, shutil, gc, random, pathlib
from pathlib import Path
import numpy as np
import torch
from PIL import Image
from datasets import load_dataset
from torch.utils.data import DataLoader
from tqdm.auto import tqdm

from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

try:
    import bitsandbytes as bnb
except Exception as e:
    bnb = None

# COCO metrics
try:
    from pycocoevalcap.eval import COCOEvalCap
    from pycocotools.coco import COCO
except Exception as e:
    COCOEvalCap = None
    COCO = None

from torchmetrics.text.bleu import BLEUScore
import sacrebleu

try:
    from codecarbon import EmissionsTracker
except Exception:
    EmissionsTracker = None

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


### (2) Configuration Setup

This section defines a `RunConfig` class to store all the configuration parameters for the experiment. The default parameters are set, such as the model ID, dataset name, batch size, number of workers, and image size. Additionally, options for energy tracking and reproducibility (via a fixed seed) are included.

The `RunConfig` is instantiated as `CFG` for use throughout the notebook.

Finally, the script ensures that the output directory exists by creating it if necessary.


In [ ]:
from dataclasses import dataclass
from pathlib import Path
import os, json as _json

@dataclass
class RunConfig:
    model_id: str = "Qwen/Qwen2-VL-2B-Instruct"
    run_name: str = "qwen2vl_coco2017"
    run_dataset: str = "coco2017"
    eval_batch_size: int = 8
    num_workers: int = 2
    image_max_side: int = 1024
    output_dir: str = "./outputs"
    save_predictions: bool = True
    track_energy: bool = True
    seed: int = 42
    max_samples: int | None = 5000

class ResizeMaxSide:
  def __init__(self, max_side=1024):
      self.max_side = max_side
  def __call__(self, img: Image.Image) -> Image.Image:
      w, h = img.size
      s = self.max_side / max(w, h)
      if s >= 1.0:
          return img
      return img.resize((int(round(w*s)), int(round(h*s))), Image.BICUBIC)

CFG = RunConfig()
Path(CFG.output_dir).mkdir(parents=True, exist_ok=True)
print(CFG)

### (3) Dataset Preparation and Loading


**COCO Dataset Download and Extraction**:
   - The script downloads the COCO validation images (`val2017.zip`) and annotations (`annotations_trainval2017.zip`)

**Dataset Class (`Coco2017ValCaptions`)**:
   - A custom dataset class is defined for COCO, which returns a dictionary containing an image, its ID, and corresponding captions.
   - Captions are pre-loaded for efficiency.

**Dataset Selection**:
   - If `run_dataset` is set to `"coco2017"`


In [ ]:
import os, zipfile, json, random
from pathlib import Path
import requests
from PIL import Image
from tqdm.auto import tqdm
from pycocotools.coco import COCO
import torch


resize_transform = ResizeMaxSide(CFG.image_max_side)

COCO_DIR = Path("/content/data/coco")
COCO_VAL_DIR = COCO_DIR / "val2017"
COCO_ANN_DIR = COCO_DIR / "annotations"
COCO_CAPTIONS_JSON = COCO_ANN_DIR / "captions_val2017.json"

VAL_ZIP_URL = "http://images.cocodataset.org/zips/val2017.zip"
ANN_ZIP_URL = "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"

def _download(url: str, dest: Path):
    dest.parent.mkdir(parents=True, exist_ok=True)
    if dest.exists():
        return
    print(f"Downloading {url} -> {dest}")
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        total = int(r.headers.get("content-length", 0))
        with open(dest, "wb") as f, tqdm(total=total, unit="B", unit_scale=True) as pbar:
            for chunk in r.iter_content(chunk_size=1 << 15):
                if chunk:
                    f.write(chunk)
                    pbar.update(len(chunk))

def _unzip(zip_path: Path, to_dir: Path, members: list[str] | None = None):
    print(f"Extracting {zip_path} -> {to_dir}")
    to_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, 'r') as z:
        if members is None:
            z.extractall(to_dir)
        else:
            for m in members:
                z.extract(m, to_dir)

def ensure_coco_val():
    # Images
    if not COCO_VAL_DIR.exists() or not any(COCO_VAL_DIR.iterdir()):
        zpath = COCO_DIR / "val2017.zip"
        _download(VAL_ZIP_URL, zpath)
        _unzip(zpath, COCO_DIR)
    # Captions annotation
    if not COCO_CAPTIONS_JSON.exists():
        zpath = COCO_DIR / "annotations_trainval2017.zip"
        _download(ANN_ZIP_URL, zpath)
        _unzip(zpath, COCO_DIR, members=["annotations/captions_val2017.json"])

class Coco2017ValCaptions(torch.utils.data.Dataset):
    """Returns dict(image: PIL.Image, image_id: int, references: list[str])"""
    def __init__(self, root_dir: Path, captions_json: Path, max_samples: int | None = None, seed: int = 42):
        super().__init__()
        self.root_dir = Path(root_dir)
        self.coco = COCO(str(captions_json))
        self.img_ids = sorted(self.coco.getImgIds())
        if max_samples is not None:
            random.seed(seed)
            self.img_ids = random.sample(self.img_ids, min(max_samples, len(self.img_ids)))
        self.refs = {}
        for img_id in self.img_ids:
            ann_ids = self.coco.getAnnIds(imgIds=[img_id])
            anns = self.coco.loadAnns(ann_ids)
            self.refs[img_id] = [a["caption"] for a in anns if "caption" in a]

    def __len__(self): return len(self.img_ids)

    def __getitem__(self, idx):
        img_id = self.img_ids[idx]
        info = self.coco.loadImgs([img_id])[0]
        img_path = self.root_dir / info["file_name"]
        with Image.open(img_path) as im:
            im = im.convert("RGB")
            im = resize_transform(im)
        return {"image": im, "image_id": int(img_id), "references": self.refs.get(img_id, [])}

# ---- Build dataset ----
if CFG.run_dataset == "coco2017":
    ensure_coco_val()
    dataset = Coco2017ValCaptions(COCO_VAL_DIR, COCO_CAPTIONS_JSON, max_samples=CFG.max_samples, seed=42)
    indices_path = Path(CFG.output_dir) / f"image_ids_{CFG.run_name}_{CFG.run_dataset}.json"
    with open(indices_path, "w", encoding="utf-8") as f:
        json.dump(getattr(dataset, "img_ids", []), f)
    print("Saved image ids to:", indices_path)
else:
    raise ValueError("Unknown dataset key")

print(f"Loaded COCO val2017 samples: {len(dataset)}")

### (4) Performance and Memory Tracking

This section defines utility functions and context managers for tracking model performance, memory usage, and energy consumption.

**`get_model_disk_size_bytes`**:
   - Computes the total disk size of a model (in bytes)

**`bytes_to_mib`**:
   - Converts bytes to MiB (Mebibytes) for easier readability of memory sizes.

**`cuda_memory_tracker`**:
   - A context manager that tracks GPU memory usage during model inference. It resets and synchronizes memory stats before and after execution, printing the peak VRAM usage and elapsed time.

**`ThroughputMeter`**:
   - A class to track throughput (images per second) and latency (in milliseconds). It accumulates data during processing and computes performance metrics like average and 95th percentile latency.

**`maybe_energy_tracker`**:
   - A context manager that tracks the energy consumption of the experiment if the `EmissionsTracker` is available and enabled.

**`cuda_memory_tracker_record`**:
   - Another context manager for tracking GPU memory usage, specifically recording the peak memory allocated and elapsed time.

In [ ]:
from contextlib import contextmanager
import time as _time


def get_model_disk_size_bytes(model_id_or_path: str) -> int:
    try:
        if os.path.isdir(model_id_or_path):
            base = Path(model_id_or_path)
        else:
            from huggingface_hub import snapshot_download
            cache_dir = snapshot_download(model_id_or_path, allow_patterns=["*.bin","*.pt","*.safetensors","*.json"], local_files_only=True)
            base = Path(cache_dir)
        total = 0
        for p in base.rglob("*"):
            if p.is_file():
                total += p.stat().st_size
        return total
    except Exception:
        return -1

def bytes_to_mib(x): return x / (1024 * 1024)

@contextmanager
def cuda_memory_tracker():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()
    try:
        yield
    finally:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        elapsed = time.perf_counter() - start
        peak = bytes_to_mib(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else -1
        print(f"[Perf] Elapsed: {elapsed:.2f}s | Peak VRAM: {peak:.0f} MiB")

class ThroughputMeter:
    def __init__(self):
        self.total_imgs = 0; self.total_time = 0.0; self.latencies = []
    def update(self, n, t):
        self.total_imgs += n; self.total_time += t; self.latencies.append(t/max(n,1))
    def report(self):
        thr = self.total_imgs / max(self.total_time, 1e-9)
        import numpy as np
        lat_ms = float(np.mean(self.latencies))*1000 if self.latencies else float("nan")
        p95_ms = float(np.percentile(self.latencies,95))*1000 if self.latencies else float("nan")
        return {"throughput_img_s": thr, "avg_latency_ms": lat_ms, "p95_latency_ms": p95_ms}

@contextmanager
def maybe_energy_tracker(enabled: bool, output_dir: str, run_tag: str):
    tracker = None
    if enabled and EmissionsTracker is not None:
        tracker = EmissionsTracker(output_dir=output_dir, output_file=f"codecarbon_{run_tag}.csv")
        tracker.start()
    try:
        yield
    finally:
        if tracker is not None:
            tracker.stop()

@contextmanager
def cuda_memory_tracker_record():
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    _t0 = _time.perf_counter()
    try:
        yield
    finally:
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        _elapsed = _time.perf_counter() - _t0
        try:
            _peak = torch.cuda.max_memory_allocated() if torch.cuda.is_available() else -1
            print(f"[Perf] Elapsed: {_elapsed:.2f}s | Peak VRAM: {bytes_to_mib(_peak) if _peak!=-1 else -1:.0f} MiB")
        except Exception as _e:
            print("[Perf] Could not read peak VRAM:", _e)


### (5) Model Loaders and Quantization

This section defines functions for loading models with different precision configurations and applying quantization techniques.

**Model Loaders**:
   - **`load_model_fp16`**: Loads a model in FP16 precision and its corresponding processor.
   - **`load_model_llm_int8`**: Loads a model with the LLM (Language Model) parts quantized to 8-bit.
   - **`load_model_llm_int8_plus_vision_int8`**: Loads a full FP16 model directly to GPU, then applies selective INT8 quantization to both the LLM and vision towers.

**Quantization**:
   - **`quantize_vision_tower_8bit`**: Quantizes only the attention and MLP layers of the vision tower (vision model), keeping other layers in FP16.
   - **`inspect_vision_quantisation`**: A utility to inspect which layers of the vision tower have been successfully quantized to INT8 and confirms that other layers remain in FP16.
   - **`quantize_llm_8bit`**: Quantizes the Linear layers of the LLM tower to 8-bit, focusing on the transformer internals.


In [ ]:
from transformers import AutoProcessor, AutoModelForVision2Seq, BitsAndBytesConfig
import bitsandbytes as bnb
import torch
import torch.nn as nn

TOKENIZER_KW = {}
GEN_KW = dict(max_new_tokens=64)
DEVICE = "auto"

def _print_footprint(model):
    try:
        mem_bytes = model.get_memory_footprint()
        mem_mib = mem_bytes / (1024 ** 2)
        print(f"[Model footprint] Runtime model size: {mem_mib:.2f} MiB")
    except Exception as e:
        print(f"[Model footprint] Could not determine size: {e}")

def load_model_fp16(model_id: str):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map=DEVICE,
        trust_remote_code=True,
    )
    _print_footprint(model)
    return processor, model

def load_model_llm_int8(model_id: str):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)
    bnb_cfg = BitsAndBytesConfig(load_in_8bit=True)

    model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        quantization_config=bnb_cfg,
        device_map=DEVICE,
        trust_remote_code=True,
    )
    _print_footprint(model)
    return processor, model

# --- Quantization passes ---
def quantize_vision_tower_8bit(model: nn.Module):
    """
    Turn the model's vision tower into INT8
    - Quantize only attention + MLP Linear layers
    - Keep patch_embed, LayerNorms, and projection in FP16
    """
    count = 0
    skipped = 0

    candidate_roots = []
    for path in ["visual", "model.visual", "model.vision_model", "vision_tower", "vision_model"]:
        obj = model
        ok = True
        for p in path.split("."):
            if hasattr(obj, p):
                obj = getattr(obj, p)
            else:
                ok = False
                break
        if ok:
            candidate_roots.append(obj)

    modules_to_visit = candidate_roots or [model]

    for root in modules_to_visit:
        for name, module in list(root.named_modules()):
            if isinstance(module, nn.Linear):
                lname = name.lower()
                if any(k in lname for k in ["attn", "attention", "mlp", "ffn", "fc1", "fc2"]):
                    parent = root
                    parts = name.split(".")
                    for p in parts[:-1]:
                        parent = getattr(parent, p)
                    old = getattr(parent, parts[-1])

                    new_lin = bnb.nn.Linear8bitLt(
                        old.in_features, old.out_features,
                        bias=old.bias is not None, has_fp16_weights=True
                    ).to(old.weight.device)

                    with torch.no_grad():
                        new_lin.weight.data.copy_(old.weight.detach().to(torch.float16).contiguous())
                        if old.bias is not None:
                            new_lin.bias.data.copy_(old.bias.detach().to(torch.float16).contiguous())

                    setattr(parent, parts[-1], new_lin)
                    count += 1
                else:
                    skipped += 1

    print(f"Quantised {count} safe vision Linear layers (attention/MLP); skipped {skipped} others.")
    _print_footprint(model)
    return model

def inspect_vision_quantisation(model: nn.Module):
    print("\n=== Vision Tower Quantisation Summary ===")
    total_linear = 0
    quantised = 0
    for name, module in model.named_modules():
        if isinstance(module, nn.Linear):
            total_linear += 1
            if isinstance(module, bnb.nn.Linear8bitLt):
                quantised += 1
                print(f"INT8:  {name}")
    print(f"---\nQuantised {quantised}/{total_linear} Linear layers total\n")

    # Confirm other layers still float
    for name, module in model.named_modules():
        lname = name.lower()
        if any(k in lname for k in ["patch_embed", "norm", "proj", "projection"]):
            # Might not have .weight (e.g., LayerNorm in some impls), guard it
            dtype = getattr(getattr(module, "weight", None), "dtype", "N/A")
            print(f"FP16 layer kept: {name} ({module.__class__.__name__}) dtype={dtype}")

def quantize_llm_8bit(model: nn.Module):
    if bnb is None:
        print("bitsandbytes not available; skipping LLM quantization")
        return model

    llm_root = None
    for path in ["language_model", "model.language_model", "text_model"]:
        obj = model
        ok = True
        for p in path.split("."):
            if hasattr(obj, p):
                obj = getattr(obj, p)
            else:
                ok = False
                break
        if ok:
            llm_root = obj
            print(f"Found LLM root at: {path}")
            break

    if llm_root is None:
        print("Could not find LLM root, skipping LLM quantization.")
        return model

    count = 0
    skipped = 0
    for name, module in list(llm_root.named_modules()):
        if isinstance(module, nn.Linear):
            lname = name.lower()
            if any(k in lname for k in ["attn", "attention", "mlp", "ffn", "fc1", "fc2", "self_attn",
                                        "k_proj", "v_proj", "q_proj", "o_proj"]):
                parent = llm_root
                parts = name.split(".")
                for p in parts[:-1]:
                    parent = getattr(parent, p)
                old = getattr(parent, parts[-1])

                new_lin = bnb.nn.Linear8bitLt(
                    old.in_features, old.out_features,
                    bias=old.bias is not None, has_fp16_weights=True
                ).to(old.weight.device)

                with torch.no_grad():
                    new_lin.weight.data.copy_(old.weight.detach().to(torch.float16).contiguous())
                    if old.bias is not None:
                        new_lin.bias.data.copy_(old.bias.detach().to(torch.float16).contiguous())

                setattr(parent, parts[-1], new_lin)
                count += 1
            else:
                skipped += 1

    print(f"Quantised {count} safe LLM Linear layers; skipped {skipped} others.")
    _print_footprint(model)
    return model

# --- Combined loader + manual quantization ---
def load_model_llm_int8_plus_vision_int8(model_id: str):
    processor = AutoProcessor.from_pretrained(model_id, trust_remote_code=True)

    print(f"Loading FULL FP16 model directly to {DEVICE} with low_cpu_mem_usage...")
    model = AutoModelForVision2Seq.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        trust_remote_code=True,
        low_cpu_mem_usage=True,
        device_map=DEVICE,
    )

    print("Model loaded. Now manually quantizing (in-place)...")

    print("Manually quantizing LLM...")
    model = quantize_llm_8bit(model)

    print("Manually quantizing vision tower...")
    model = quantize_vision_tower_8bit(model)

    print("Model fully modified.")
    _print_footprint(model)
    return processor, model


### (6) Collate & Generation Helpers

**`collate_batch`**:
   - This function is used to process batches of data for training or inference. It extracts the **images**, **image IDs**, and **references** from the input batch and returns them as a dictionary.

**`generate_captions`**:
   - This function generates captions for a list of images using a pretrained model. It works by creating a **multimodal chat prompt** for each image and sending it to the model.
   - The captions are then **decoded** and **cleaned**:
     - **Assistant’s response** is extracted.
     - **Placeholders like `<image>`** are removed.
     - Whitespace is collapsed to ensure a concise caption.


In [ ]:
import re

def collate_batch(batch):
    images = [b["image"] for b in batch]
    image_ids = [b["image_id"] for b in batch]
    references = [b["references"] for b in batch]
    return {"images": images, "image_ids": image_ids, "references": references}

@torch.no_grad()
def generate_captions(processor, model, images):
    texts, imgs = [], []
    for img in images:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": "Describe the image in one concise sentence. Output only the caption."},
                ],
            }
        ]
        prompt = processor.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,  # adds assistant turn
        )
        texts.append(prompt)
        imgs.append(img)

    inputs = processor(
        text=texts,
        images=imgs,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    outputs = model.generate(**inputs, **GEN_KW)
    decoded = processor.batch_decode(outputs, skip_special_tokens=True)

    cleaned = []
    for d in decoded:
        s = d

        # 1) Keep only the assistant’s reply if a chat transcript leaked in
        if "\nassistant\n" in s:
            s = s.split("\nassistant\n", 1)[1]

        # 2) Drop any leftover role headers and the image placeholder
        s = s.replace("<image>", "")
        s = re.sub(r'^(system|user)\n.*?(?=\nassistant\n|$)', '', s, flags=re.S)

        # 3) Collapse whitespace and keep a single concise line
        s = " ".join(s.strip().split())

        cleaned.append(s)

    return cleaned


### (7) Evaluation: BLEU-4 (sacrebleu) + Spotcheck

This section defines helper functions for evaluating the model's performance using **BLEU-4** and for saving Spotcheck examples.

**`compute_bleu4_sacre` Function**:
   - Computes the **BLEU-4 score** using the `sacrebleu` library, which is designed for fast, reproducible evaluation of machine translation and captioning models.
   - **`preds`**: List of generated captions.
   - **`refs_list`**: List of reference captions (one or more per image).
   - Returns the BLEU-4 score, which measures how well the predicted captions match the reference captions.

**`save_spotcheck_samples` Function**:
   - Randomly selects a subset of **25 samples** from a `.jsonl` file (or a specified number of samples) for manual spot-checking.
   - Saves the selected samples to a new `.jsonl` file and returns the output path.


In [ ]:
from pycocotools.coco import COCO
from pycocoevalcap.cider.cider import Cider
import json
import sacrebleu

def compute_bleu4_sacre(preds, refs_list):
    # sacrebleu expects refs as list[list_of_refs_per_system_out]
    bleu = sacrebleu.corpus_bleu(
        preds,
        list(zip(*refs_list)) if len(refs_list) and isinstance(refs_list[0], list) else refs_list
    )
    return bleu.score

def save_spotcheck_samples(jsonl_path: str, k: int = 25):
    import json, random
    lines = open(jsonl_path, "r", encoding="utf-8").read().strip().splitlines()
    random.seed(SEED)
    sample = random.sample(lines, min(k, len(lines)))
    out_path = jsonl_path.replace(".jsonl", "_spotcheck25.jsonl")
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("\n".join(sample))
    return out_path



## (8) Runner

In [ ]:
from collections import defaultdict

def run_experiment(tag: str, loader_fn):
    print(f"\n========== Running {tag} ==========")
    predictions_path = os.path.join(CFG.output_dir, f"{CFG.run_name}_{CFG.run_dataset}_{tag}.jsonl")
    os.makedirs(CFG.output_dir, exist_ok=True)
    processor, model = loader_fn(CFG.model_id)
    model.eval()

    # ---- Model size on disk (cache) ----
    try:
        from huggingface_hub import snapshot_download
        cache_dir = snapshot_download(CFG.model_id, local_files_only=True)
        size_bytes = get_model_disk_size_bytes(cache_dir)
    except Exception:
        size_bytes = -1

    # ---- DataLoader & meter ----
    dl = DataLoader(
        dataset,
        batch_size=CFG.eval_batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        collate_fn=collate_batch,
    )
    meter = ThroughputMeter()
    all_preds, sys_out, refs_flat = [], [], []

    # ==== Inference-only tracking start ====
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

    with maybe_energy_tracker(CFG.track_energy, CFG.output_dir, tag):
        with cuda_memory_tracker_record():
            for batch in tqdm(dl, desc=f"{tag}"):
                t0 = time.perf_counter()
                caps = generate_captions(processor, model, batch["images"])
                if torch.cuda.is_available():
                    torch.cuda.synchronize()
                t1 = time.perf_counter()
                meter.update(len(caps), t1 - t0)

                for iid, cap, refs in zip(batch["image_ids"], caps, batch["references"]):
                    rec = {"image_id": int(iid), "caption": cap, "references": refs}
                    all_preds.append(rec)
                    sys_out.append(cap)
                    refs_flat.append(refs)
    # ==== Inference-only tracking end ====

    peak_vram_mib = bytes_to_mib(torch.cuda.max_memory_allocated()) if torch.cuda.is_available() else -1

    if CFG.save_predictions:
        with open(predictions_path, "w", encoding="utf-8") as f:
            for rec in all_preds:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    perf = meter.report()
    print("Perf:", perf)

    # ==== Metrics ====
    bleu4 = compute_bleu4_sacre(sys_out, refs_flat)
    print(f"BLEU-4 (sacrebleu): {bleu4:.2f}")

    cider = None
    try:
        # Construct res and gts from all_preds and dataset.refs
        res = {rec["image_id"]: [rec["caption"]] for rec in all_preds}
        img_ids = set(res.keys()) # Get the image ids from the predictions
        gts = {iid: [ref for ref in dataset.refs.get(iid, [])] for iid in img_ids}

        cider_scorer = Cider()
        cider, _ = cider_scorer.compute_score(gts, res)
        print(f"CIDEr: {cider:.3f}")
    except Exception as e:
        print("[Warn] CIDEr computation failed:", e)
        cider = None

    results = {
        "tag": tag,
        "dataset": CFG.run_dataset,
        "num_samples": len(dataset),
        "throughput_img_s": perf.get("throughput_img_s"),
        "avg_latency_ms": perf.get("avg_latency_ms"),
        "p95_latency_ms": perf.get("p95_latency_ms"),
        "model_disk_size_bytes": size_bytes,
        "BLEU4": bleu4,
        "CIDEr": cider,
        "peak_vram_mib": peak_vram_mib,
        # NOTE: energy_kwh_*, emissions_kgco2, duration_s, avg_power_w
    }

    res_path = os.path.join(CFG.output_dir, f"metrics_{CFG.run_name}_{CFG.run_dataset}_{tag}.json")
    with open(res_path, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2)

    spot_path = save_spotcheck_samples(predictions_path, 25)
    print("Saved:", res_path)
    print("25-sample spot-check:", spot_path)
    return results


## (9) Execute Basline FP16



In [ ]:
all_results = []

all_results.append(run_experiment("fp16", load_model_fp16))

In [ ]:
print("\n=== Summary ===")
print(all_results[-1])

!head -n 5 ./outputs/qwen2vl_coco2017_fp16_spotcheck25.jsonl

## (10) Execute Weight-only 8-bit

In [ ]:
all_results = []

all_results.append(run_experiment("llm_int8", load_model_llm_int8))

In [ ]:
print("\n=== Summary ===")
print(all_results[-1])

!head -n 5 ./outputs/qwen2vl_coco2017_llm_int8_spotcheck25.jsonl

## (11) Execute selective quantize vision tower to INT8


In [ ]:
all_results = []

all_results.append(run_experiment("llm_int8_vision_int8", load_model_llm_int8_plus_vision_int8))


In [ ]:
print("\n=== Summary ===")
print(all_results[-1])

!head -n 5 ./outputs/qwen2vl_coco2017_llm_int8_vision_spotcheck25.jsonl

## results and hardware

### (12) Save hardware info

In [ ]:
import platform, subprocess, torch, json, os, sys

env_info = {
    "python": platform.python_version(),
    "pytorch": torch.__version__,
    "cuda_available": torch.cuda.is_available(),
    "cuda_device_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    "gpu_count": torch.cuda.device_count() if torch.cuda.is_available() else 0,
}

try:
    pip_freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True).splitlines()
except Exception as e:
    pip_freeze = [f"pip_freeze_error: {e}"]

env_info["pip_freeze"] = pip_freeze

outp = os.path.join(CFG.output_dir, f"env_{CFG.run_name}_{CFG.run_dataset}.json")
with open(outp, "w", encoding="utf-8") as f:
    json.dump(env_info, f, indent=2)
print("Saved env/hardware to:", outp)


### (13) Compile results

In [ ]:
import glob, json, csv, os

pattern = os.path.join(CFG.output_dir, f"metrics_{CFG.run_name}_{CFG.run_dataset}_*.json")
rows = []
for p in sorted(glob.glob(pattern)):
    with open(p, "r", encoding="utf-8") as f:
        rows.append(json.load(f))

if not rows:
    print("No metrics found at pattern:", pattern)
else:
    keys = sorted({k for r in rows for k in r.keys()})
    out_csv = os.path.join(CFG.output_dir, f"summary_{CFG.run_name}_{CFG.run_dataset}.csv")
    with open(out_csv, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        w.writerows(rows)
    print("Wrote summary CSV:", out_csv)
    print("Included runs:", [r.get("tag") for r in rows])


### (14) Print results

In [ ]:
import os, glob, csv
import pandas as pd


# Helper to extract tag from filename
def _tag_from_path(p: str) -> str:
    base = os.path.basename(p)
    name, _ = os.path.splitext(base)
    return name.split("_", 1)[1] if "_" in name else name.split("-", 1)[1]


def load_all_codecarbon(out_dir: str, preview_each: bool = False):
    paths = sorted(
        set(glob.glob(os.path.join(out_dir, "codecarbon_*.csv"))) |
        set(glob.glob(os.path.join(out_dir, "codecarbon-*.csv")))
    )
    if not paths:
        print("No CodeCarbon CSVs found in:", out_dir)
        return None, None

    summaries = []
    frames = {}
    for p in paths:
        tag = _tag_from_path(p)
        try:
            df = pd.read_csv(p)
            frames[tag] = df

            if df.empty:
                print(f"[Warn] CodeCarbon file is empty: {p}")
                continue

            last_row = df.iloc[-1]

            duration_s = last_row.get("duration", 0.0)
            energy_kwh_total = last_row.get("energy_consumed", 0.0)
            energy_kwh_cpu = last_row.get("cpu_energy", 0.0)
            energy_kwh_gpu = last_row.get("gpu_energy", 0.0)
            energy_kwh_ram = last_row.get("ram_energy", 0.0)
            emissions_kgco2 = last_row.get("emissions", 0.0)

            avg_power_w = (energy_kwh_total * 1000.0) / (duration_s / 3600.0) if duration_s > 0 else 0.0

            row = {
                "tag": tag,
                "cc_file": p,
                "duration_s": duration_s,
                "energy_kwh_total": energy_kwh_total,
                "energy_kwh_cpu": energy_kwh_cpu,
                "energy_kwh_gpu": energy_kwh_gpu,
                "energy_kwh_ram": energy_kwh_ram,
                "emissions_kgco2": emissions_kgco2,
                "avg_power_w": avg_power_w,
            }
            summaries.append(row)

            if preview_each:
                print(f"\n=== {tag} — {os.path.basename(p)} ===")
                display(df.head(5))
                display(df.tail(3))
        except Exception as e:
            print(f"[Error] Could not process CodeCarbon file {p}: {e}")


    if not summaries:
        print("No valid CodeCarbon data processed.")
        return None, frames

    summary_df = pd.DataFrame(summaries).sort_values("energy_kwh_total", ascending=False)
    print("CodeCarbon summary across experiments:")
    display(summary_df)

    return summary_df, frames


summary_df, cc_frames = load_all_codecarbon(CFG.output_dir, preview_each=False)

summary_path = "./outputs/summary_qwen2vl_coco2017_coco2017.csv"
try:
    df = pd.read_csv(summary_path)
    display(df.head(10))
except FileNotFoundError:
    print(f"Summary CSV not found at {summary_path}. Run the experiment cells first.")

# (15) Show 3 example images with generated captions and image-level BLEU4 & CIDEr

In [ ]:
import os, json, random
from pathlib import Path
from typing import List, Dict, Optional
import numpy as np
import matplotlib.pyplot as plt
import sacrebleu


OUTPUT_DIR = Path(CFG.output_dir)

# ---------- COCO-aware image resolver ----------
_COCO_ID2FILE = None
_COCO_SPLIT_DIR = None
_COCO_ROOT_CANDIDATES: List[Path] = []

def _init_coco_mapping():
    """Build mapping {image_id: file_name} from COCO annotations and prepare root candidates."""
    global _COCO_ID2FILE, _COCO_SPLIT_DIR, _COCO_ROOT_CANDIDATES
    if _COCO_ID2FILE is not None:
        return

    id2file, split_dir = {}, None
    ann_path = None
    try:
        ann_path = Path(str(COCO_CAPTIONS_JSON))
    except NameError:
        ann_path = None

    if ann_path and ann_path.is_file():
        with open(ann_path, "r", encoding="utf-8") as f:
            data = json.load(f)
        id2file = {int(img["id"]): img["file_name"] for img in data.get("images", [])}
        stem = ann_path.stem.lower()
        if "val2017" in stem:
            split_dir = "val2017"
        elif "train2017" in stem:
            split_dir = "train2017"

        if split_dir:
            _COCO_ROOT_CANDIDATES.extend([
                ann_path.parent.parent / split_dir,
                ann_path.parent.parent,
            ])
    else:
        id2file, split_dir = {}, None

    for key in ["coco_images_dir", "images_dir", "img_root", "coco_root"]:
        if hasattr(CFG, key):
            val = getattr(CFG, key)
            if val:
                _COCO_ROOT_CANDIDATES.insert(0, Path(val))

    env_root = os.environ.get("COCO_IMAGES_DIR")
    if env_root:
        _COCO_ROOT_CANDIDATES.append(Path(env_root))

    # Deduplicate
    seen = set()
    uniq = []
    for p in _COCO_ROOT_CANDIDATES:
        try:
            rp = p.resolve()
        except Exception:
            rp = p
        if rp not in seen:
            seen.add(rp)
            uniq.append(rp)
    _COCO_ROOT_CANDIDATES = uniq

    _COCO_ID2FILE = id2file
    _COCO_SPLIT_DIR = split_dir

def _resolve_coco_image_path(image_id: int) -> Optional[str]:
    _init_coco_mapping()

    file_name = None
    if _COCO_ID2FILE and int(image_id) in _COCO_ID2FILE:
        file_name = _COCO_ID2FILE[int(image_id)]

    try:
        if isinstance(dataset, dict) and "id2path" in dataset and image_id in dataset["id2path"]:
            p = dataset["id2path"][image_id]
            return p if os.path.exists(p) else None
        if hasattr(dataset, "id2path"):
            id2path = getattr(dataset, "id2path")
            if isinstance(id2path, dict) and image_id in id2path:
                p = id2path[image_id]
                return p if os.path.exists(p) else None
        if isinstance(dataset, list):
            for item in dataset:
                if isinstance(item, dict) and int(item.get("image_id", -1)) == int(image_id):
                    if "image_path" in item and os.path.exists(item["image_path"]):
                        return item["image_path"]
                    if "file_name" in item and "root" in item:
                        p = os.path.join(item["root"], item["file_name"])
                        return p if os.path.exists(p) else None
    except NameError:
        pass

    rel_candidates = []
    if file_name:
        if _COCO_SPLIT_DIR:
            rel_candidates.append(Path(_COCO_SPLIT_DIR) / file_name)
        rel_candidates.append(Path(file_name))

    for root in _COCO_ROOT_CANDIDATES:
        for rel in rel_candidates:
            p = root / rel
            if p.exists():
                return str(p)

    for root in _COCO_ROOT_CANDIDATES:
        for split in ["val2017", "train2017", "images", "imgs"]:
            for pat in [f"{image_id}.jpg", f"{image_id}.png", f"{image_id:012d}.jpg", f"{image_id:012d}.png"]:
                p = root / split / pat
                if p.exists():
                    return str(p)
            for pat in [f"{image_id}.jpg", f"{image_id}.png", f"{image_id:012d}.jpg", f"{image_id:012d}.png"]:
                p = root / pat
                if p.exists():
                    return str(p)

    return None

# ---------- Predictions & scoring ----------
def _find_latest_predictions_file() -> Path:
    files = sorted(
        OUTPUT_DIR.glob(f"{CFG.run_name}_{CFG.run_dataset}_*.jsonl"),
        key=lambda p: p.stat().st_mtime,
        reverse=True
    )
    if not files:
        raise FileNotFoundError(f"No predictions JSONL files found in {OUTPUT_DIR}.")
    return files[0]

def _load_predictions(pred_path: Path) -> List[Dict]:
    preds = []
    with open(pred_path, "r", encoding="utf-8") as f:
        for line in f:
            preds.append(json.loads(line))
    return preds

# Load most recent predictions
pred_file = _find_latest_predictions_file()
all_recs = _load_predictions(pred_file)
if not all_recs:
    raise RuntimeError(f"No records in predictions file: {pred_file}")

# Pick 3 unique images
unique_ids = list({rec["image_id"] for rec in all_recs})
random.seed(1234)
random.shuffle(unique_ids)
picked_ids = unique_ids[:3]
picked = [next(rec for rec in all_recs if rec["image_id"] == iid) for iid in picked_ids]

# Show and score
for i, rec in enumerate(picked, 1):
    iid = int(rec["image_id"])
    hypo = rec["caption"]
    refs = rec.get("references", []) or []

    bleu4 = sacrebleu.sentence_bleu(hypo, refs).score if refs else float("nan")

    print(f"\nExample {i} — image_id={iid}")
    print("Caption:", hypo)
    if refs:
        print("Ref[0]:", refs[0])

    img_path = _resolve_coco_image_path(iid)
    if img_path and os.path.exists(img_path):
        img = plt.imread(img_path)
        plt.figure()
        plt.imshow(img)
        plt.axis("off")
        plt.title(f"ID {iid}  |  BLEU4={bleu4:.2f}  ")
        plt.show()
    else:
        print("[Note] Could not resolve image path for display. Tried candidates near COCO annotations and CFG roots.")
        print(f"BLEU4={bleu4:.2f}  ")

print("\nUsed predictions file:", pred_file)


# (16) Compare experiments + make key charts


In [ ]:
import os, json
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

OUTPUT_DIR = Path(CFG.output_dir)

def _gather_metrics_json() -> pd.DataFrame:
    rows = []
    for p in OUTPUT_DIR.glob(f"metrics_{CFG.run_name}_{CFG.run_dataset}_*.json"):
        with open(p, "r", encoding="utf-8") as f:
            data = json.load(f)
        tag = data.get("tag") or p.stem.split("_")[-1]
        data["tag"] = str(tag)
        rows.append(data)
    if not rows:
        raise FileNotFoundError(f"No metrics_*.json files in {OUTPUT_DIR}")
    return pd.DataFrame(rows)

metrics_df = _gather_metrics_json()

cc_summary, _cc_frames = load_all_codecarbon(CFG.output_dir, preview_each=False)

def _normalize_tag(t: str) -> str:
    t = str(t)
    for suf in ["_infer_only", "-infer-only", "_inference", "-inference"]:
        if t.endswith(suf):
            return t[: -len(suf)]
    return t

if cc_summary is not None and not cc_summary.empty:
    cc_norm = cc_summary.copy()
    cc_norm["tag"] = cc_norm["tag"].map(_normalize_tag)
    cc_norm = cc_norm.drop(columns=["cc_file"], errors="ignore")
    merged = metrics_df.merge(cc_norm, on="tag", how="left", suffixes=("", "_cc"))
else:
    merged = metrics_df.copy()

order = ["fp16", "llm_int8", "llm_int8_vision_int8"]
merged = merged[merged["tag"].isin(order)].copy()
merged = merged.sort_values("tag", key=lambda s: s.map({k:i for i,k in enumerate(order)}))

numeric_cols = merged.select_dtypes(include=[np.number]).columns.tolist()
comparison_df = merged[["tag"] + numeric_cols].set_index("tag").reindex(order)

pd.options.display.float_format = "{:.6f}".format
print("All numeric variables per experiment:")
display(comparison_df)

have_cpu = "energy_kwh_cpu" in comparison_df.columns
have_gpu = "energy_kwh_gpu" in comparison_df.columns
have_ram = "energy_kwh_ram" in comparison_df.columns
if have_cpu and have_gpu and have_ram:
    xs = order
    cpu = pd.to_numeric(comparison_df["energy_kwh_cpu"], errors="coerce").fillna(0).values
    gpu = pd.to_numeric(comparison_df["energy_kwh_gpu"], errors="coerce").fillna(0).values
    ram = pd.to_numeric(comparison_df["energy_kwh_ram"], errors="coerce").fillna(0).values

    plt.figure()
    plt.bar(xs, cpu, label="CPU")
    plt.bar(xs, gpu, bottom=cpu, label="GPU")
    plt.bar(xs, ram, bottom=cpu+gpu, label="RAM")
    plt.title("Energy by Component (kWh) per Experiment")
    plt.xlabel("Experiment Tag")
    plt.ylabel("kWh")
    plt.xticks(rotation=0)
    plt.legend()
    plt.tight_layout()
    plt.show()
else:
    print("Stacked energy plot skipped: missing energy_kwh_cpu/gpu/ram.")


def _pretty_title(col: str) -> str:
    return col.replace("_", " ").title()

for col in numeric_cols:
    ys = pd.to_numeric(comparison_df[col], errors="coerce")
    if ys.isna().all():
        continue
    plt.figure()
    plt.bar(order, ys.reindex(order).fillna(0.0).values)
    plt.title(f"{_pretty_title(col)} — Comparison")
    plt.xlabel("Experiment Tag")
    plt.ylabel(col)
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
